# Publish Triangle Mesh from Multipatch Shapefile

This notebook shows how you can sign in and publish a Triangle Mesh geoscience object from a shapefile containing only multipatch shapes to your chosen Evo workspace.

**Important:** This notebook requires Python 3.10, 3.11, or 3.12 (not 3.14+) due to evo.notebooks dependencies.

In the first cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In Cell 2, replace `"your-client-id"` with your Evo app client ID before running the cell.

In [ ]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(client_id="your-client-id").login()

C:\Users\Maitri.Anvekar\AppData\Roaming\Python\Python310\site-packages\evo\notebooks\authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

## Convert and Publish Shapefile to Triangular Mesh

In the cells below we:
1. Read the organization, hub, and workspace from the widget selections above
2. Specify the path to our shapefile
3. Add optional tags
4. Call `convert_shp` to convert and publish

The shapefile will be:
- Translated to vertices, triangles, and shapes which are stored in parquet files
- Formatted according to the triangle-mesh schema
- Published to your evo workspace

In [2]:
# Install local package into this notebook kernel (run once per environment)
%pip install -e ../..

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///C:/Users/Maitri.Anvekar/Documents/evo_data_converters_PR/evo-data-converters/packages/shp
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for evo-data-converters-shp (pyproject.toml): started
  Building editable for evo-data-converters-shp (pyproject.toml): finished with status 'done'
  Created wheel for evo-data-converters-shp:


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Get workspace from widget selections
service_mgr = manager._service_manager
org = service_mgr.get_current_organization()
hub = service_mgr.get_current_hub()
workspace = service_mgr.get_current_workspace()
selected_workspace_id = workspace.id

# Path to your shapefile (can be any of the individual files or just the extensionless name)
# Using relative path from notebook location
filename = "../../tests/data/test_shape.shp"

# PRJ File (Optional)
filename_prj = "../../tests/data/test_shape.prj"

# Tags to add to the geoscience object
tags = {"Source": "Jupyter Notebook", "Type": "Shapefile Triangular Mesh"}

# Upload path in Evo workspace
upload_path = "shapefile/imports"

print(f"Organization: {org.display_name}")
print(f"Hub: {hub.display_name}")
print(f"Workspace: {workspace.display_name} (ID: {selected_workspace_id})")
print(f"Shapefile: {Path(filename).name}")
print("\nConverting and publishing...")

# Convert and publish
results = convert_shp(
    filepath=filename,
    filepath_prj=filename_prj,
    tags=tags,
    evo_workspace_metadata=None,
    service_manager_widget=manager,
    upload_path=upload_path,
    publish_objects=True,
    overwrite_existing_objects=True,
)

# Print results
print("\n✅ Successfully published!")
print("\nPublished objects:")
for obj_metadata in results:
    print(f"  - {obj_metadata.name}")
    print(f"    ID: {obj_metadata.id}")

Organization: Evo Platform Success
Hub: Australia East
Workspace: maitestmcp (ID: 1850bf93-ea96-4a91-945f-10aa73527d88)
Shapefile: test_shape.shp

Converting and publishing...
Publishing Shapefile

✅ Successfully published!

Published objects:
  - test_shape.json
    ID: dd4f20ee-e891-4739-afc6-7258fe3570d4


## Convert Without Publishing (for testing)

You can also convert the Shapefile without publishing to inspect the resulting object:

In [4]:
import json

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Convert but don't publish
shp_objects = convert_shp(
    filepath="../../tests/data/test_shape.shp",
    filepath_prj="../../tests/data/test_shape.prj",
    upload_path="./parquet_output",
    publish_objects=False,  # Don't publish
)

# Inspect the mesh object
shp = shp_objects[0]
print(f"Shape name: {shp.name}")

shp_dict = shp.as_dict()  # Use as_dict() instead of to_dict()
print("\nJSON representation:")
print(json.dumps(shp_dict, indent=2))

Shape name: test_shape

JSON representation:
{
  "triangles": {
    "vertices": {
      "data": "e6790507ed3aa54736c2a947143effdaebf35f491d7987e344ebee76ba701b8e",
      "length": 772,
      "width": 3,
      "data_type": "float64"
    },
    "indices": {
      "data": "c8a1dad0ec40bfc1dde9d308e718607aed9fe7ae66ec9adb9b5c54873886a6b5",
      "length": 3080,
      "width": 3,
      "data_type": "uint64"
    }
  },
  "parts": {
    "attributes": [
      {
        "name": "name",
        "key": "ba7d2379d6744bc271ea9982a70ff056a63a3c380a383033ee7b5f0d18ba2ae6",
        "attribute_type": "string",
        "values": {
          "data": "ba7d2379d6744bc271ea9982a70ff056a63a3c380a383033ee7b5f0d18ba2ae6",
          "length": 2,
          "data_type": "string"
        }
      }
    ],
    "chunks": {
      "data": "e6edc3aeb14f57c48652205e4ace4b959faa594f0fcac21bc884b3d7d83cefe8",
      "length": 2,
      "width": 2,
      "data_type": "uint64"
    }
  },
  "name": "test_shape",
  "uuid": null,